In [ ]:
source_path = "/Volumes/workspace/gagealspach/ams_demo_source/gateway_source"
staging_data_path = "/Volumes/workspace/gagealspach/ams_demo_staging/data"
checkpoint_path = "/Volumes/workspace/gagealspach/ams_demo_staging/checkpoint"

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)
from pyspark.sql.functions import col, current_timestamp


source_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("location", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("_source_version", LongType(), True),
    StructField("_operation", StringType(), True),
    StructField("_source_updated_ts", TimestampType(), True),
])


gateway_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(source_schema)
    .load(source_path)
    .withColumn("_gateway_source_file", col("_metadata.file_name"))
    .withColumn("_gateway_captured_ts", current_timestamp())
)

gateway_query = (
    gateway_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(staging_data_path)
)
gateway_query.awaitTermination()